# monty_demo — Reasoning Layer for Physical-AI Teleop Data

**Pitch artifact for [usemonty.com](https://usemonty.com)** — *real-world data infrastructure for physical AI*.

> _"Teach physical AI from human motion — the trajectories AND the world knowledge the human was implicitly using. Past teleop becomes data + priors; every new task starts with both, and the system gets sharper with every attempt."_

This notebook walks the full pipeline end-to-end on real LeRobot ALOHA data. Three prior coffee-brewing episodes seed the substrate; three subsequent "new attempts" — cross-skill, cross-embodiment, and an outlier — exercise the self-improvement loop. The merged human-priors / data-driven impedance recommendation appears at the end.


In [ ]:
from monty_demo import (
    Episode,
    KnowledgeGraph,
    ingest,
    reason,
    print_brief_diff,
)
from monty_demo._timing import format_timing_table, reset_timings
import numpy as np
from dataclasses import replace as dc_replace
from monty_demo.schemas import EpisodeSource

reset_timings()
kg = KnowledgeGraph()

## 1. Seed the substrate — three prior brew-coffee episodes

Each `ingest()` runs the full pipeline: load → estimate `k_hat` → segment phases → infer intent / objects / skills (from `REPO_METADATA`) → outlier check → write to KG.

In [ ]:
for i in range(3):
    ep = Episode.from_lerobot('lerobot/aloha_static_coffee', index=i)
    populated = ingest(kg, ep)
    print(f'  ingested ep {i}: {populated.n_frames:>4} frames @ {populated.source.fps} fps   '
          f'phases={len(populated.phases or [])}   intent={populated.intent.name}')

print()
print('KG state:', kg.stats())

## 2. Reason about a new task (BEFORE any new data)

Target: brew coffee on the same ALOHA bimanual rig, with `mug` as the target object. The brief should be supported by 3 matched episodes, with the human prior on `mug` (`fragility=moderate`, `safety_context=["contains_liquid"]`, `suggested_impedance="gentle"`) already merged into the recommended impedance regime.

In [ ]:
brief_before = reason(
    kg,
    intent='brew-coffee',
    target_objects=('mug', 'coffee-machine'),
    embodiment='aloha-bimanual',
)

print('matched_episodes:           ', brief_before.matched_episodes)
print('confidence:                 ', brief_before.confidence)
print('embodiment_diversity:       ', brief_before.embodiment_diversity)
print('recommended_impedance:      ', brief_before.recommended_impedance_regime)
print('safety_warnings:            ', brief_before.safety_warnings)
print('suggested_skills:           ', brief_before.suggested_skills)
print()
print('plan:')
for p in brief_before.plan:
    print(f'  {p.name:<10} ~{p.expected_duration_s:.2f}s   k_hat band {p.suggested_k_hat_band}   '
          f'(supported by {p.n_supporting_episodes} episodes)')
print()
print('notes:')
for n in brief_before.notes:
    print('  -', n)

## 3. Self-improvement loop

Three deliberately distinct "new attempts" land one at a time. Each demonstrates a different reasoning capability:

1. **Cross-skill transfer** — different intent, shared dexterity
2. **Cross-embodiment calibration** — same intent, different arm; confidence should *drop* despite more data
3. **Outlier detection** — anomalous contact phase, surfaced as a warning

Together: transfer + calibration + critical sense.

### Attempt 1 — cross-skill (velcro threading)

Different intent (`thread-velcro`), different objects, but shares `fine-bimanual-coordination` skill. Shouldn't increase `matched_episodes` for `brew-coffee`, but should surface in `transferable_skills_observed`.

In [ ]:
ep = Episode.from_lerobot('lerobot/aloha_static_thread_velcro', index=0)
ingest(kg, ep)

brief_after_1 = reason(kg, intent='brew-coffee', target_objects=('mug', 'coffee-machine'), embodiment='aloha-bimanual')
print('--- diff after Attempt 1 ---')
print_brief_diff(brief_before, brief_after_1, kg)

### Attempt 2 — cross-embodiment (Koch single-arm)

Same intent (hand-labeled `brew-coffee` for the demo), but a single-arm Koch instead of bimanual ALOHA. The reasoner's cross-embodiment penalty *0.6 on the score and the *0.85 confidence dip should net out to **lower confidence** despite more matches — because cross-embodiment evidence is less reliable, not more.

In [ ]:
ep = Episode.from_lerobot('lerobot/koch_pick_place_5_lego', index=0)
ingest(kg, ep)

brief_after_2 = reason(kg, intent='brew-coffee', target_objects=('mug', 'coffee-machine'), embodiment='aloha-bimanual')
print('--- diff after Attempt 2 ---')
print_brief_diff(brief_after_1, brief_after_2, kg)
print()
print(f'embodiment_diversity went {brief_after_1.embodiment_diversity} → {brief_after_2.embodiment_diversity}')

### Attempt 3 — outlier detection

An attempt whose contact phase is dramatically longer than the established baseline. In production this would be a fumble or a partial failure flagged for review; here we synthesize one (with honest framing) by extending a coffee episode's contact span. The reasoner should flag it via z-score against existing contact-phase durations.

In [ ]:
# Honest framing: in production this is a real attempt that segments anomalously.
# Here we synthesize one to demonstrate the capability deterministically.
real_ep = Episode.from_lerobot('lerobot/aloha_static_coffee', index=3)
T = real_ep.n_frames
# Extend by holding the joints frozen for 200 extra frames mid-episode (simulates a stall)
EXTRA = 200
stall_at = T // 2
extended_pos = np.concatenate([
    real_ep.joint_positions[:stall_at],
    np.repeat(real_ep.joint_positions[stall_at:stall_at+1], EXTRA, axis=0),
    real_ep.joint_positions[stall_at:],
], axis=0).astype(np.float32)
extended_act = np.concatenate([
    real_ep.joint_actions[:stall_at],
    np.linspace(real_ep.joint_actions[stall_at], real_ep.joint_actions[stall_at]+0.5, EXTRA).astype(np.float32),
    real_ep.joint_actions[stall_at:],
], axis=0).astype(np.float32)
v = np.linalg.norm(np.diff(extended_pos, axis=0), axis=1) / real_ep.dt
ee_v = np.concatenate([[v[0]], v]).astype(np.float32)
outlier_ep = Episode(
    source=EpisodeSource(repo_id='lerobot/aloha_static_coffee', index=999, embodiment='aloha-bimanual', fps=real_ep.source.fps),
    n_frames=extended_pos.shape[0], dt=real_ep.dt,
    joint_positions=extended_pos, joint_actions=extended_act,
    ee_velocity_norm=ee_v,
)
ingest(kg, outlier_ep)

brief_final = reason(kg, intent='brew-coffee', target_objects=('mug', 'coffee-machine'), embodiment='aloha-bimanual')
print('--- diff after Attempt 3 (outlier) ---')
print_brief_diff(brief_after_2, brief_final, kg)

## 4. Cumulative diff — start to finish

What did the system *learn* across the three new attempts?

In [ ]:
print('=== brief_before vs brief_final ===')
print_brief_diff(brief_before, brief_final)
print()
print('confidence:', brief_before.confidence, '→', brief_final.confidence)
print('matched:   ', len(brief_before.matched_episodes), '→', len(brief_final.matched_episodes))
print('embodiments:', brief_before.embodiment_diversity, '→', brief_final.embodiment_diversity)

## 5. Human priors in action

The brief surfaces both the data-driven `k_hat` band (from contact-phase aggregation across matched episodes) and the **merged** `recommended_impedance_regime` after applying the human's prior knowledge of the target objects. The data alone might say "around 0.3–0.6"; the prior on `mug` (`fragility=moderate`, `contains_liquid`, `suggested_impedance=gentle`) tightens it down.

In [ ]:
print('object_knowledge:')
for ok in brief_final.object_knowledge:
    print(f'  {ok.name:<16} fragility={ok.fragility:<10} mass={ok.mass_category:<6} '
          f'safety={ok.safety_context}  prior_impedance={ok.suggested_impedance}')
print()
contact_band = next((p.suggested_k_hat_band for p in brief_final.plan if p.name == 'contact'), None)
print(f'data-driven contact k_hat band:    {contact_band}')
print(f'recommended_impedance_regime:     {brief_final.recommended_impedance_regime}')
print()
print('safety_warnings:')
for w in brief_final.safety_warnings:
    print('  -', w)

## 6. Cypher export — the migration path

NetworkX is the in-memory backend; `kg.to_cypher()` emits the equivalent Cypher for the same query, signalling the path to a real graph DB (Kuzu / Neo4j) at scale.

In [ ]:
print(kg.to_cypher({'intent': 'brew-coffee', 'embodiment': 'aloha-bimanual', 'phase': 'contact', 'min_k_hat': 0.2}))

## 7. Timing — speed as evidence

Every public op self-times. This run's actuals:

In [ ]:
print(format_timing_table())

## What this enables

- **Every new task starts with retrieved priors** instead of cold — phase plan, stiffness band, suggested skills, object-aware safety.
- **Cross-task transfer is structural, not just label-based** — `transferable_skills_observed` surfaces dexterity learned in adjacent tasks.
- **Bad new data is flagged before it pollutes downstream training** — z-score outlier detection on every ingest.
- **Human knowledge of the world is first-class** — fragility, mass class, safety context, suggested impedance, all merged into the brief alongside the data-driven signal.
- **Confidence is calibrated** — adding cross-embodiment evidence *lowers* confidence, not raises it. A reasoner that blindly grows confidence with more data is broken.
